# 🔐 CyberNews Scraper — Google Colab

Pipeline para recopilar, puntuar y distribuir las **4 noticias más relevantes de ciberseguridad** del mes.

### Fuentes
The Hacker News · SecurityWeek · Kaspersky Securelist · WeLiveSecurity ES

### Pasos
1. **Paso 1** – Cargar el código desde GitHub
2. **Paso 2** – Instalar dependencias
3. **Paso 3** – Ajustar parámetros de ejecución
4. **Paso 4** – Ejecutar el pipeline
5. **Paso 5** – Ver resultados en el notebook
6. **Paso 6** – Descargar archivos _(opcional)_

> El envío por email / Teams / Slack se configura en una fase posterior.


In [ ]:
#@title 🔧 Paso 1 – Clonar / sincronizar el proyecto desde GitHub

import os
import sys
import subprocess
from pathlib import Path

REPO_URL    = 'https://github.com/dalesos92/cybernews_scraper.git'
PROJECT_DIR = Path('/content/cybernews-scraper')

def _current_commit(path):
    r = subprocess.run(['git', '-C', str(path), 'rev-parse', 'HEAD'],
                       capture_output=True, text=True)
    return r.stdout.strip()[:7] if r.returncode == 0 else None

os.chdir('/content')

if not PROJECT_DIR.exists():
    # 1ª ejecución: clonar
    print('Clonando repositorio...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(PROJECT_DIR)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Clonado fallido. Revisa el mensaje de arriba.')
    print(result.stderr.strip())
    _action = 'clonado'
else:
    # Ya existe: comparar commit local con HEAD remoto
    local_commit = _current_commit(PROJECT_DIR)
    r = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'ls-remote', 'origin', 'HEAD'],
        capture_output=True, text=True,
    )
    remote_commit = r.stdout.split()[0][:7] if r.returncode == 0 and r.stdout.strip() else None

    if remote_commit and local_commit != remote_commit:
        print(f'⚠ Versión local ({local_commit}) desactualizada → remoto ({remote_commit}). Sincronizando...')
        # fetch + reset --hard: siempre funciona aunque haya cambios locales (.env, output/, etc.)
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', 'main'], check=True, capture_output=True)
        reset = subprocess.run(
            ['git', '-C', str(PROJECT_DIR), 'reset', '--hard', 'origin/main'],
            capture_output=True, text=True,
        )
        if reset.returncode != 0:
            print(reset.stderr)
            raise RuntimeError('git reset --hard fallido: ' + reset.stderr[:200])
        print(reset.stdout.strip())
        _action = 'actualizado'
    else:
        _action = 'ya al día'

# ── Configurar entorno Python ──
os.chdir(str(PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# ── Versión canónica: se lee del archivo VERSION en el repo ──
_commit      = _current_commit(PROJECT_DIR)
_version_file = PROJECT_DIR / 'VERSION'
_expected    = _version_file.read_text(encoding='utf-8').strip() if _version_file.exists() else '?'
_up_to_date  = _commit == _expected

print(f'✓ Repo {_action}')
print(f'  Commit  : {_commit} {"✅ última versión" if _up_to_date else "⚠ versión distinta a la esperada"}')
print(f'  Esperado: {_expected} (según archivo VERSION)')
print(f'  Directorio: {os.getcwd()}')


In [ ]:
#@title 📦 Paso 2 – Instalar dependencias

import subprocess
import os
from pathlib import Path

# Localizar requirements.txt: primero en el cwd establecido por el Paso 1,
# luego en la ruta fija de extracción, y finalmente en el directorio del notebook.
def _find_req():
    candidates = [
        Path(os.getcwd()) / 'requirements.txt',
        Path('/content/cybernews-scraper/requirements.txt'),
    ]
    try:
        candidates.insert(0, Path(PROJECT_DIR) / 'requirements.txt')
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return None

_req = _find_req()
if _req is None:
    raise FileNotFoundError(
        'No se encontró requirements.txt. '
        'Asegúrate de haber ejecutado el Paso 1 antes.'
    )

print(f'✓ requirements.txt encontrado en: {_req}')
print('Instalando dependencias...')
result = subprocess.run(
    ['pip', 'install', '-q', '-r', str(_req)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print('⚠ Advertencias / errores durante la instalación:')
    print((result.stderr or result.stdout)[-2000:])
else:
    print('✓ Dependencias instaladas correctamente')


In [ ]:
#@title ⚙️ Paso 3 – Parámetros de ejecución

#@markdown #### Contenido
top_n = 4  #@param {type:"integer"}
lookback_days = 30  #@param {type:"integer"}

#@markdown #### Plantilla de email
EMAIL_TEMPLATE = 'c'  #@param ["a", "b", "c"]

#@markdown #### Fuentes activas (solo en español)
enable_welivesecurity = False   #@param {type:"boolean"}
enable_cybersecnews_es = True  #@param {type:"boolean"}
enable_hispasec = True  #@param {type:"boolean"}
enable_revista_ciberseguridad = True  #@param {type:"boolean"}
enable_incibe = False  #@param {type:"boolean"}
enable_seguinfo = True  #@param {type:"boolean"}
enable_kaspersky_latam = True  #@param {type:"boolean"}

print(f'✓ top_n={top_n} | lookback_days={lookback_days}')
_fuentes = [
    ('WeLiveSec',       enable_welivesecurity),
    ('CyberSecNewsES',  enable_cybersecnews_es),
    ('Hispasec',        enable_hispasec),
    ('RevistaCiber',    enable_revista_ciberseguridad),
    ('INCIBE',          enable_incibe),
    ('Segu-Info',       enable_seguinfo),
    ('KasperskyLATAM',  enable_kaspersky_latam),
]

print('  Fuentes: ' + ', '.join(f'{n}={"ON" if v else "off"}' for n, v in _fuentes))

In [ ]:
#@title 🚀 Paso 4 – Ejecutar pipeline

# Leer GOOGLE_APPSCRIPT_WEBHOOK_URL desde Colab Secrets (opcional)
try:
    from google.colab import userdata as _userdata4
    _webhook_url = _userdata4.get('GOOGLE_APPSCRIPT_WEBHOOK_URL') or ''
except Exception:
    _webhook_url = ''


import os
from pathlib import Path

# Generar .env solo con las variables necesarias para ver noticias
_env_vars = {
    'LOG_LEVEL':                     'INFO',
    'OUTPUT_DIR':                    'output',
    'DB_PATH':                       'data/cybernews.db',
    'HTTP_TIMEOUT':                  '30',
    'HTTP_MAX_RETRIES':              '3',
    'TOP_N':                         str(top_n),
    'LOOKBACK_DAYS':                 str(lookback_days),
    'ENABLE_WELIVESECURITY':         str(enable_welivesecurity).lower(),
    'ENABLE_CYBERSECNEWS_ES':        str(enable_cybersecnews_es).lower(),
    'ENABLE_HISPASEC':               str(enable_hispasec).lower(),
    'ENABLE_REVISTA_CIBERSEGURIDAD': str(enable_revista_ciberseguridad).lower(),
    'ENABLE_INCIBE':                 str(enable_incibe).lower(),
    'ENABLE_SEGUINFO':               str(enable_seguinfo).lower(),
    'ENABLE_KASPERSKY_LATAM':        str(enable_kaspersky_latam).lower(),
    'GOOGLE_EMAIL_TEMPLATE':         EMAIL_TEMPLATE,
    'GOOGLE_APPSCRIPT_WEBHOOK_URL':  _webhook_url,
}
Path('.env').write_text('\n'.join(f'{k}={v}' for k, v in _env_vars.items()), encoding='utf-8')
print('✓ .env generado')
print('=' * 60)

# --dry-run: genera archivos de salida pero no guarda en BD ni envía nada
os.system('python main.py --dry-run')


In [ ]:
#@title 📊 Paso 5 – Ver resultados

import json
from pathlib import Path
from IPython.display import HTML, Markdown, display

_out = Path('output')

# Resumen en texto
_json_path = _out / 'top4_monthly.json'
if _json_path.exists():
    _data = json.loads(_json_path.read_text(encoding='utf-8'))
    print(f"Generado : {_data['generated_at']}")
    print(f"Asunto   : {_data['subject']}")
    print()
    for item in _data['items']:
        print(f"  #{item['rank']} [{item['source']}]")
        print(f"     {item['title']}")
        print(f"     {item['url']}")
        print()
else:
    print('⚠ top4_monthly.json no encontrado. ¿Ejecutaste el Paso 4?')

# Markdown renderizado
_md_path = _out / 'top4_monthly.md'
if _md_path.exists():
    print('\n' + '─' * 50)
    display(Markdown(_md_path.read_text(encoding='utf-8')))

# Preview HTML del email
_html_path = _out / 'top4_email.html'
if _html_path.exists():
    print('\n' + '─' * 50)
    display(HTML(_html_path.read_text(encoding='utf-8')))


In [ ]:
#@title 💾 Paso 6 – Descargar archivos de salida (opcional)

from pathlib import Path

try:
    from google.colab import files as colab_files
    _found = 0
    for _fname in ['top4_monthly.json', 'top4_monthly.md', 'top4_email.html']:
        _fpath = Path('output') / _fname
        if _fpath.exists():
            colab_files.download(str(_fpath))
            print(f'✓ Descargando {_fname}')
            _found += 1
    if _found == 0:
        print('⚠ No se encontraron archivos. Ejecuta el Paso 4 primero.')
except ImportError:
    print('ℹ Esta celda solo funciona en Google Colab.')


In [ ]:
#@title ☁️ Paso 7 – Subir a Drive y enviar correo por Apps Script

#@markdown Requiere tener configurados en **Colab Secrets** (🔑):
#@markdown - `GOOGLE_DRIVE_FOLDER_ID` — ID de la carpeta Google Drive
#@markdown - `GOOGLE_APPSCRIPT_TOKEN` — token generado con `setupWebhookToken()`
#@markdown - `GOOGLE_APPSCRIPT_WEBHOOK_URL` — URL completa del Web App de Apps Script

EMAIL_TEMPLATE        = 'c'  # a | b | c

import importlib.util, sys
from pathlib import Path

# ── Leer secretos desde Colab Secrets ──
try:
    from google.colab import userdata
    DRIVE_FOLDER_ID = userdata.get('GOOGLE_DRIVE_FOLDER_ID')
    APPSCRIPT_TOKEN = userdata.get('GOOGLE_APPSCRIPT_TOKEN')
    APPSCRIPT_WEBHOOK_URL = userdata.get('GOOGLE_APPSCRIPT_WEBHOOK_URL') or ''
except Exception as e:
    raise RuntimeError(
        'No se pudieron leer los secretos. '
        'Agrega GOOGLE_DRIVE_FOLDER_ID y GOOGLE_APPSCRIPT_TOKEN en el panel 🔑.'
    ) from e

if not DRIVE_FOLDER_ID:
    raise ValueError('GOOGLE_DRIVE_FOLDER_ID está vacío en Colab Secrets.')

# ── Cargar drive_uploader directamente desde el archivo en disco ──
# Evita cualquier problema de caché en sys.modules o __pycache__.
_module_path = Path('/content/cybernews-scraper/src/drive_uploader.py')
if not _module_path.exists():
    raise FileNotFoundError(f'No se encontró {_module_path}. Ejecuta el Paso 1.')

_spec = importlib.util.spec_from_file_location('_drive_uploader_fresh', str(_module_path))
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
upload_artifacts = _mod.upload_artifacts

# Verificar que el parámetro credentials existe
import inspect
_sig = inspect.signature(upload_artifacts)
if 'credentials' not in _sig.parameters:
    raise ImportError(
        'drive_uploader.py en disco no tiene el parámetro credentials. '
        'Ejecuta: !git -C /content/cybernews-scraper pull'
    )
print(f'✓ drive_uploader cargado desde disco (parámetros: {list(_sig.parameters)})')

# ── Autenticar con la cuenta Google del usuario de Colab ──
from google.colab import auth as _colab_auth
_colab_auth.authenticate_user()

import google.auth
_creds, _ = google.auth.default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/userinfo.email',
])
print('✓ Autenticación Google completada')

# Refrescar token para obtener access_token válido
from google.auth.transport.requests import Request as _GoogleRequest
_creds.refresh(_GoogleRequest())
_auth_header = {'Authorization': f'Bearer {_creds.token}'}

# ── HTML a subir (generado por el pipeline con el template seleccionado) ──
_html_path = Path('output/top4_email.html')

_json_path = Path('output/top4_monthly.json')
_paths = [p for p in [_html_path, _json_path] if p.exists()]

if not _paths:
    raise FileNotFoundError('No hay archivos de salida. Ejecuta el Paso 4 primero.')

# ── Subir a Google Drive ──
print('Subiendo archivos a Google Drive...')
ids = upload_artifacts(paths=_paths, folder_id=DRIVE_FOLDER_ID, credentials=_creds)
print(f"  Carpeta Drive: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print("  ^ Este ID debe coincidir con CONFIG.DRIVE_FOLDER_ID en Apps Script")
for name, fid in ids.items():
    print(f'  ✓ {name}  →  https://drive.google.com/file/d/{fid}/view')

# ── Disparar Apps Script ──
if APPSCRIPT_WEBHOOK_URL.startswith('https://'):
    import httpx

    if '/dev' in APPSCRIPT_WEBHOOK_URL:
        print('\n❌ ERROR: La URL contiene "/dev" — usa la URL de producción (/exec)')
    else:
        print('\nLlamando al Web App de Apps Script...')
        try:
            # Apps Script 302 → httpx convierte POST a GET automáticamente (correcto)
            resp = httpx.post(
                APPSCRIPT_WEBHOOK_URL,
                json={'token': APPSCRIPT_TOKEN or ''},
                headers=_auth_header,
                timeout=30,
                follow_redirects=True,
            )
            if resp.status_code == 401:
                print('❌ 401: el Web App rechazó la autenticación.')
                print('   Verifica que la cuenta de Colab pertenece al dominio BBVA.')
            else:
                try:
                    data = resp.json()
                    if data.get('status') == 200:
                        print(f"  ✓ {data['message']}")
                    else:
                        print(f"  ⚠ Apps Script error {data.get('status')}: {data.get('message')}")
                except Exception:
                    print(f'  Respuesta inesperada (HTTP {resp.status_code}): {resp.text[:300]}')
        except Exception as _ex:
            print(f'  Error de red: {_ex}')
else:
    print('\n⚠ APPSCRIPT_WEBHOOK_URL no configurada. Archivos subidos a Drive.')
    print('   Disparar manualmente desde el Sheet: 🔒 CyberNews → Enviar correo del mes')
